# 06 — Whole-ROI Kinetic Model Fitting

This notebook fits DCE-MRI pharmacokinetic models to a whole-ROI mean concentration curve.

Instead of fitting each voxel independently, the signal across all ROI voxels is pooled
into a single mean curve and the kinetic model is fit to that curve.

**Contents**
1. Setup and data loading
2. ROI inspection - visual check for file uploads and correct masking
3. AIF inspection - computes AIF concentration time curve and visual check
4. Lesion aggregation — computes lesion concentration and visual check
5. Model fitting with AIF time delay — continuous optimisation


## 1. Setup and data loading

In [42]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from dce_mri.config import (
    AcquisitionConfig, SubjectConfig,
    FittingConfig, RoiFittingConfig,
    AifConfig
)

from dce_mri.kinetic_models import TCXM, GKM, tcxm_ct_analytic, gkm_ct
from dce_mri.signal_models import (
    signal_to_concentration, concentration_to_signal, shift_aif, extract_aif
)
from dce_mri.fitting_roi import (
    fit_roi,
    fit_roi_with_delay_grid,
    fit_roi_with_delay_continuous,
    fit_multi_roi,
    _aggregate_roi_signal,
)

%matplotlib qt
plt.rcParams['figure.dpi'] = 120

In [122]:
# ──: change parameters ──────────

acq     = AcquisitionConfig(TR=4.87, FA=10.0, r1=3.9,nbase=slice(0,8)) #nbase will change with spokes_per_frame
subject = SubjectConfig("test", T10=1530.0, B1=1.0)
cfg     = FittingConfig(n_lhs=200, n_top=10)
spokes_per_frame = 8

# ──: load real data ─────────────
#  edit these for each subject/dataset you load 
from dce_mri.volume import load_nifti_canonical
from dce_mri.io import load_aif
    
DCE_PATH  = "/mnt/labspace/users/allison/fastMRI_data/tight_masks/canonical/FMB084/MGRASP_temptv_FMB084_lambda_0_003.nii.gz"#"/mnt/labspace/users/allison/fastMRI_data/GRASP_NIfTI/fastMRI_breast_084_2_8spf_NII/slice_000_frame_000.nii.gz"#"/mnt/labspace/users/allison/fastMRI_data/tight_masks/canonical/FMB081/MGRASP_temptv_FMB081_lambda_0_003.nii.gz"#"/mnt/labspace/users/allison/fastMRI_data/GRASP_NIfTI/fastMRI_breast_013_2_8spf_NII/slice_000_frame_000.nii.gz"#"/mnt/labspace/users/allison/fastMRI_data/tight_masks/canonical/FMB013/MGRASP_temptv_FMB013_lambda_0_003.nii.gz"# 
AIF_PATH  = "/mnt/labspace/users/allison/fastMRI_data/tight_masks/canonical/FMB084/Aorta_RotOrtho_5_Case84.nii"
MASK_PATH = "/mnt/labspace/users/allison/fastMRI_data/tight_masks/canonical/FMB084/Lesion 2_RotOrtho_5_Case84_tight.nii"
subject = SubjectConfig(
    subject_id         = "FMB084",
    T10                = 1530.0,
    B1                 = 1.0,
    dce_path_override  = DCE_PATH,
    aif_path_override   = AIF_PATH,
    mask_path_override  = MASK_PATH,
)
    
sig_4d, ref_img = load_nifti_canonical(subject.dce_path)

# Time parameters
nspoke    = 288    # total spokes across the whole scan
scantime  = 150     # total scan duration, seconds
spoke_res = spokes_per_frame     # spokes per frame, change per case

dt = scantime * spoke_res / nspoke
t = (np.arange(sig_4d.shape[-1])) * dt
    
# load masks — binary ROI
mask_aif,aif_nift      = load_nifti_canonical(subject.aif_path)
aif_ROI         = mask_aif.astype(bool)
mask_data, _ = load_nifti_canonical(subject.mask_path)
lesion_ROI   = mask_data.astype(bool)
    
# optional: load segmentation for multi-ROI fitting
# seg_vol, _ = load_nifti_canonical(subject.seg_path)

print(f"Loaded: {sig_4d.shape} volume, {lesion_ROI.sum()} tumor voxels, {aif_ROI.sum()} aif voxels")


Loaded: (320, 320, 192, 36) volume, 89 tumor voxels, 81 aif voxels


# 2. ROI inspection

In [123]:
# ──: change indexing based on ROI ──────────
aif_idx = np.where(aif_ROI.sum(axis=(0, 1)) > 0)[0][0]
print("Aorta slice:", aif_idx)

from scipy.ndimage import binary_dilation, shift, binary_erosion

mask    = aif_ROI[:, :, aif_idx]      # original mask
shift_n = 0                            # pixels to shift left

# dilate first
structure  = np.ones((3, 3))  # 8-connectivity; use default (cross) for 4-connectivity
grown_mask = binary_dilation(mask, structure=structure, iterations=2)

# then shift the dilated mask
shifted = shift(grown_mask.astype(float), shift=[-shift_n, 0], order=0) > 0.5

aif_ROI[:, :, aif_idx] = shifted


lesion_sum_per_slice = lesion_ROI.sum(axis=(0, 1))
z_indices_lesions = np.where(lesion_sum_per_slice > 0)[0]
print("All lesion slices:", z_indices_lesions)
lesion_idx =np.argmax(lesion_sum_per_slice)
print("Max lesion slice:",lesion_idx)

time_idx = 15
    
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
img_aif = sig_4d[:, :, aif_idx, time_idx]
img_lesion = sig_4d[:, :, lesion_idx, time_idx]
  
axes[0].imshow(img_aif.T, cmap="gray", origin="lower")
axes[0].imshow(aif_ROI[:,:,aif_idx].T,cmap="Reds", alpha= 0.5 * aif_ROI[:,:,aif_idx].T ,origin="lower")
axes[0].set_title(f"Aorta ROI (z={aif_idx})")
axes[0].axis("off")

axes[1].imshow(img_lesion.T,cmap="gray", origin="lower")
axes[1].imshow(lesion_ROI[:,:,lesion_idx].T,cmap="Reds",alpha=0.5 * lesion_ROI[:,:,lesion_idx].T ,origin="lower")
axes[1].set_title(f"Lesion ROI (z={lesion_idx})")
axes[1].axis("off")
    
plt.suptitle(f"{subject.subject_id}, t={time_idx}")
plt.tight_layout()
plt.show()

Aorta slice: 110
All lesion slices: [ 96  97  98  99 100 101]
Max lesion slice: 98


## 3. AIF time intensity curve inspection

In [124]:
from dce_mri.config import AifConfig
from dce_mri.signal_models import extract_aif

aif_cfg = AifConfig(
    T10        = 1779.0,
    hematocrit = 0.4,
    top_frac   = 1.0,
    nbase      = acq.nbase,
)
#aif_ROI[:, :, aif_idx + 1] = aif_ROI[:, :, aif_idx]
#aif_ROI[:, :, aif_idx - 1] = aif_ROI[:, :, aif_idx]

aif, aif_signal = extract_aif(sig_4d, aif_ROI, acq, aif_cfg, B1=subject.B1)

print(f"AIF signal range : [{aif_signal.min():.1f}, {aif_signal.max():.1f}]")
print(f"AIF plasma range : [{aif.min():.4f}, {aif.max():.4f}] mM")
print(f"AIF signal: {aif_signal}")
print(f"AIF concentration: {aif_signal}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t, aif_signal, 'o-')
axes[0].set_title("Mean aorta signal")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Signal Intensity(a.u)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, aif, 'o-', color='crimson')
axes[1].set_title("AIF (plasma, hematocrit-corrected)")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Concentration (mM)")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


print(f"AIF peak time  : {t[np.argmax(aif)]:.1f} s")
print(f"AIF peak value : {aif.max():.4f} mM")
print(f"AIF baseline   : {aif[acq.nbase].mean():.6f} mM")

AIF signal range : [7.5, 45.6]
AIF plasma range : [-0.0565, 3.3568] mM
AIF signal: [ 9.85018887  7.53195947  8.07256607  9.68273579  8.76413981 11.2275333
 11.816059    9.01815501 14.99278766 17.88507909 22.70288309 42.18296485
 45.57313134 40.1990701  39.47966811 35.72320625 36.34465984 38.05264607
 36.62048043 34.35636885 37.34485018 36.01524899 36.25299807 39.12840791
 36.23123665 36.97182855 34.17273758 39.23668749 35.96130703 36.02619565
 39.75553266 34.06977745 36.0693668  33.95683785 37.82065608 39.51760773]
AIF concentration: [ 9.85018887  7.53195947  8.07256607  9.68273579  8.76413981 11.2275333
 11.816059    9.01815501 14.99278766 17.88507909 22.70288309 42.18296485
 45.57313134 40.1990701  39.47966811 35.72320625 36.34465984 38.05264607
 36.62048043 34.35636885 37.34485018 36.01524899 36.25299807 39.12840791
 36.23123665 36.97182855 34.17273758 39.23668749 35.96130703 36.02619565
 39.75553266 34.06977745 36.0693668  33.95683785 37.82065608 39.51760773]
AIF peak time  : 50.0 

In [125]:
import os
import nibabel as nib
aff = aif_nift.affine
header = aif_nift.header.copy()
save_path =os.path.join("/mnt/labspace/users/allison/fastMRI_data/tight_masks/canonical/FMB084/","edited_AIF_roi")
nifti_aif= nib.Nifti1Image(aif_ROI, affine=aff, header=header)
nib.save(nifti_aif, save_path)

## 4. Lesion time intensity curve inspection

In [126]:
roi_cfg_sig  = RoiFittingConfig(average_space='signal')
roi_cfg_conc = RoiFittingConfig(average_space='concentration')

Ct_sig,  n_sig,  St_sig  = _aggregate_roi_signal(
    sig_4d, lesion_ROI, acq, subject, roi_cfg_sig
)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# signal space
axes[0].plot(t, St_sig,  'b-', lw=2)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Signal (a.u.)')
axes[0].set_title('Mean Lesion signal ')
axes[0].grid(True, alpha=0.3)

# concentration space
axes[1].plot(t, Ct_sig,  'b-', lw=2)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('CA concentration (mM)')
axes[1].set_title('Mean Lesion concentration')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Signal aggregation   : {n_sig} voxels")
print(f"  Ct range: [{Ct_sig.min():.4f}, {Ct_sig.max():.4f}] mM")
print(f"  St range: [{St_sig.min():.1f}, {St_sig.max():.1f}] a.u.")

Signal aggregation   : 89 voxels
  Ct range: [-0.0072, 0.0682] mM
  St range: [108.5, 147.9] a.u.


## 5. Model fitting with AIF time delay — continuous optimisation

In [127]:
# tau bounds — adjust based on your data and expected delay range
TAU_LB = -10.0   # s
TAU_UB =  20.0   # s 
TAU_P0 =   0.0   # s — initial guess

roi_cfg = RoiFittingConfig(average_space='signal')
print(f"Continuous tau bounds: [{TAU_LB}, {TAU_UB}] s")

print("\nFitting TCXM + continuous delay ...")
result_tcxm_cont = fit_roi_with_delay_continuous(
    sig_4d, lesion_ROI, aif, t, TCXM, acq, subject, cfg, roi_cfg,
    tau_lb=TAU_LB, tau_ub=TAU_UB, tau_p0=TAU_P0, seed=0, 
    interp_method="pchip"
)

print("\n── TCXM + continuous delay ───────────────────────")
print(f"  success      : {result_tcxm_cont.success}")
print(f"  cost         : {result_tcxm_cont.best_cost:.4e}")
print(f"  best tau     : {result_tcxm_cont.best_tau:.3f} s")
for name, val in zip(TCXM.param_names, result_tcxm_cont.best_params):
    print(f"  {name:8s}: {val:.5f}")

# plot continuous delay fit
Ct_sig_mean = _aggregate_roi_signal(sig_4d, lesion_ROI, acq, subject, roi_cfg)[0]

shifted_aif_best = shift_aif(aif, result_tcxm_cont.best_tau, t, acq, method="pchip")
Ct_pred_best     = TCXM.fn(result_tcxm_cont.best_params, t, shifted_aif_best)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t, Ct_sig_mean, 'k.', ms=4, label='Ct mean (data)')
ax.plot(t, Ct_pred_best, 'r-', lw=2,
        label=f'TCXM fit (tau={result_tcxm_cont.best_tau:.2f}s)')
ax.plot(t, aif * Ct_sig_mean.max() / aif.max(),
        'b--', lw=1, alpha=0.5, label='AIF (scaled)')
ax.plot(t, shifted_aif_best * Ct_sig_mean.max() / aif.max(),
        'r--', lw=1, alpha=0.7, label='Best Shift AIF (scaled)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('CA concentration (mM)')
ax.set_title('TCXM — continuous delay fit')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Continuous tau bounds: [-10.0, 20.0] s

Fitting TCXM + continuous delay ...

── TCXM + continuous delay ───────────────────────
  success      : True
  cost         : 1.5645e-03
  best tau     : 15.910 s
  ve      : 0.03520
  vp      : 0.00552
  fp      : 0.00770
  ps      : 0.00079


In [ ]:
import nibabel as nib
img = nib.load("/mnt/labspace/users/allison/fastMRI_data/tight_masks/canonical/FMB016/MGRASP_temptv_FMB016_lambda_0_003.nii.gz")
print(img.header.extensions)
for ext in img.header.extensions:
    print(ext.get_code(), ext.get_content()[:500])  # peek at content